# Notebook 5: Causal Analysis

## 5.0 Preamble

This notebook applies **propensity score matching (PSM)** to estimate the causal effects of
driving conditions and model choices on autonomous driving performance. Rather than relying
on raw outcome comparisons, PSM constructs matched pairs of episodes that are similar on
observed covariates but differ in treatment assignment, yielding more credible effect estimates.
We analyse five treatments: rain, nighttime, urban terrain, PPO vs BC model architecture, and
comfort-optimized (chill) PPO vs BC driving style. All parameters are loaded from
`configs/causal.yaml` and the analysis is orchestrated by the `CausalAnalysisAgent`.

In [ ]:
import sys
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from scipy.spatial import cKDTree

# Ensure the project root is on sys.path for src imports
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.agents import CausalAnalysisAgent

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 5.1 Configuration

### 5.1.1 Loading Parameters

All causal analysis hyperparameters live in `configs/causal.yaml`. This keeps
the notebook free of hardcoded values and ensures that the notebook, the
`CausalAnalysisAgent`, and any downstream scripts all operate from the same
configuration. The three key scalars are the number of bootstrap resamples for
confidence intervals, the minimum treated-group size below which a treatment is
skipped, and the random seed for reproducibility.

In [ ]:
cfg = load_config("causal")

print(f"Bootstrap resamples : {cfg['n_bootstrap']}")
print(f"Min treated units   : {cfg['min_treated']}")
print(f"Random seed         : {cfg['random_seed']}")
print(f"Number of treatments: {len(cfg['treatments'])}")

### 5.1.2 Treatment Definitions

Each treatment is specified declaratively in YAML using a `{field, op, value}` DSL.
The agent reconstructs callable filter predicates from these definitions at
runtime, so the notebook never contains hardcoded group membership logic. The
table below summarises how each treatment partitions episodes into treated and
control groups, what outcome metric is measured, and which covariates are
balanced during matching.

| Treatment | Outcome | Treated Group | Control Group | Covariates |
|---|---|---|---|---|
| `rain` | route_completion | weather == HardRainNoon | weather == ClearNoon | model_type_code, town_code |
| `night` | route_completion | weather == ClearNight | weather == ClearNoon | model_type_code, town_code |
| `urban` | collision_rate | town in {Town01, Town03} | town in {Town05} | weather_code, model_type_code |
| `ppo` | route_completion | model_type == ppo | model_type == bc | weather_code, town_code |
| `driving_style` | route_completion | model_type == ppo AND driving_style == chill | model_type == bc | weather_code, town_code |

## 5.2 Causal Framework

### 5.2.1 Research Questions Mapping

The five PSM treatments correspond directly to five research questions about
how driving conditions and model architecture affect autonomous driving
performance. Each question is answered by estimating the average treatment
effect (ATE) of a specific binary treatment on the relevant outcome metric.
This mapping keeps the statistical analysis anchored to concrete, interpretable
questions.

| Research Question | PSM Treatment |
|---|---|
| Q1: Does rain reduce route completion? | `rain` |
| Q2: Does nighttime reduce route completion? | `night` |
| Q3: Does urban terrain increase collision rate? | `urban` |
| Q4: Does PPO improve route completion over BC? | `ppo` |
| Q5: Does comfort-optimized PPO reduce route completion? | `driving_style` |

### 5.2.2 Assumptions

Propensity score matching relies on a set of assumptions that must hold for the
estimated treatment effects to have a causal interpretation. In a simulator
setting some of these are more plausible than in observational field data, but
none should be taken for granted. The numbered list below catalogues every
assumption the analysis depends on.

1. **Stable Unit Treatment Value Assumption (SUTVA):** Each episode's outcome depends only on its own treatment assignment, not on other episodes' assignments.
2. **Positivity (overlap):** Every episode with a given covariate profile has a nonzero probability of appearing in both treated and control groups.
3. **Unconfoundedness (no unmeasured confounders):** Conditional on the observed covariates, treatment assignment is independent of potential outcomes.
4. **Correct propensity model specification:** The logistic regression model captures the true relationship between covariates and treatment assignment.
5. **Common support:** The propensity score distributions of treated and control groups overlap sufficiently for matching to be meaningful.
6. **No interference between episodes:** An episode's outcome is unaffected by what happened in any other episode (no carry-over effects).
7. **Treatment is binary per episode:** Each episode is either fully treated or fully control; there is no partial or graded treatment.
8. **Covariates measured pre-treatment:** All matched covariates (town, weather, model type) are fixed before the episode runs, not affected by the treatment.
9. **No post-treatment adjustment:** We do not condition on variables that could be consequences of the treatment (e.g., average speed during the episode).
10. **Balanced matching achieved (SMD < 0.1):** After matching, standardized mean differences on all covariates fall below the conventional 0.1 threshold.
11. **Sufficient sample size per treatment arm:** Each treatment has at least `min_treated` episodes in both groups to support reliable inference.
12. **Independence of matched pairs for bootstrap:** Resampled pairs in the bootstrap are treated as independent draws, valid when episodes are independent.
13. **Rosenbaum bounds valid under monotone treatment effect:** Sensitivity analysis assumes that any hidden bias shifts treatment odds monotonically.
14. **Simulator episodes independent and identically structured:** CARLA episodes use the same route length, spawn logic, and NPC density within a condition.

### 5.2.3 Estimand Definition

In causal inference, two common estimands are the **Average Treatment Effect
(ATE)** and the **Average Treatment Effect on the Treated (ATT)**. The ATE
estimates the expected difference in outcomes if the entire population were
switched from control to treatment, while the ATT restricts attention to those
units that actually received treatment. We use the ATE here because our goal is
to understand the general effect of each condition across all episodes, not just
among those that happened to be assigned to the treated group. Since the
simulator can in principle expose any episode to any condition, the ATE is the
more policy-relevant quantity: it answers "what would happen on average if we
deployed under this condition?" rather than "what happened among the episodes
that were already under this condition?"

## 5.3 Pre-Matching Diagnostics

### 5.3.1 Covariate Balance

Before performing propensity score matching, it is essential to examine how well
the treated and control groups are balanced on observed covariates. The
standardized mean difference (SMD) quantifies covariate imbalance: values above
0.1 suggest meaningful differences that matching should correct. This diagnostic
establishes the baseline imbalance that motivates PSM in the first place.

In [ ]:
# Load evaluation results
results_dir = PROJECT_ROOT / "results"
eval_path = results_dir / "eval_results.json"

with open(eval_path) as f:
    records = json.load(f)

print(f"Loaded {len(records)} evaluation episodes.")

# Encode covariates
weather_enc = LabelEncoder()
town_enc = LabelEncoder()
weather_enc.fit([r["weather"] for r in records])
town_enc.fit([r["town"] for r in records])

cov_arrays = {
    "weather_code": weather_enc.transform([r["weather"] for r in records]).astype(float),
    "town_code": town_enc.transform([r["town"] for r in records]).astype(float),
    "model_type_code": np.array([1.0 if r["model_type"] == "ppo" else 0.0 for r in records]),
}


def compute_smd(x_treated: np.ndarray, x_control: np.ndarray) -> float:
    """Standardized mean difference between two groups."""
    pooled_std = np.sqrt((x_treated.var() + x_control.var()) / 2)
    if pooled_std == 0:
        return 0.0
    return float(abs(x_treated.mean() - x_control.mean()) / pooled_std)


# Build filter callables from YAML DSL (mirrors agent logic)
def build_filter(condition):
    if isinstance(condition, list):
        filters = [build_filter(c) for c in condition]
        return lambda r: all(f(r) for f in filters)
    field, op, value = condition["field"], condition["op"], condition["value"]
    if op == "eq":
        return lambda r, _f=field, _v=value: r[_f] == _v
    elif op == "in":
        _vset = set(value)
        return lambda r, _f=field, _vs=_vset: r[_f] in _vs
    raise ValueError(f"Unknown operator: {op}")


# Compute pre-matching SMD for each treatment x covariate
smd_rows = []
for tdef in cfg["treatments"]:
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = [i for i, r in enumerate(records) if t_filter(r)]
    c_idx = [i for i, r in enumerate(records) if c_filter(r)]
    for cov_name in tdef["covariates"]:
        arr = cov_arrays[cov_name]
        smd_val = (
            compute_smd(arr[t_idx], arr[c_idx])
            if len(t_idx) > 0 and len(c_idx) > 0
            else float("nan")
        )
        smd_rows.append({
            "treatment": tdef["name"],
            "covariate": cov_name,
            "n_treated": len(t_idx),
            "n_control": len(c_idx),
            "SMD_pre_match": round(smd_val, 4),
        })

smd_df = pd.DataFrame(smd_rows)
print("\nPre-matching covariate balance (SMD):")
display(smd_df)

### 5.3.2 Propensity Score Distributions

Fitting a logistic regression on the covariates produces a propensity score for
each episode -- the predicted probability of receiving the treatment given its
covariates. Comparing the treated and control propensity score distributions
before matching reveals the degree of overlap (common support). Poor overlap
indicates that some treated episodes have no comparable controls, which would
make the ATE estimate unreliable.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, tdef in enumerate(cfg["treatments"]):
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = np.array([i for i, r in enumerate(records) if t_filter(r)])
    c_idx = np.array([i for i, r in enumerate(records) if c_filter(r)])

    if len(t_idx) < cfg["min_treated"] or len(c_idx) < cfg["min_treated"]:
        axes[idx].set_title(f"{tdef['name']} -- insufficient data")
        continue

    all_idx = np.concatenate([t_idx, c_idx])
    T = np.array([1] * len(t_idx) + [0] * len(c_idx))
    X = np.column_stack([cov_arrays[cov][all_idx] for cov in tdef["covariates"]])

    lr = LogisticRegression(C=1.0, max_iter=500, random_state=cfg["random_seed"])
    lr.fit(X, T)
    pscore = lr.predict_proba(X)[:, 1]

    ax = axes[idx]
    ax.hist(pscore[T == 1], bins=20, alpha=0.6, label="Treated", color="steelblue")
    ax.hist(pscore[T == 0], bins=20, alpha=0.6, label="Control", color="coral")
    ax.set_title(f"{tdef['name']} -- Raw Propensity Scores")
    ax.set_xlabel("Propensity Score")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

# Hide unused subplots
for j in range(len(cfg["treatments"]), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Pre-Matching Propensity Score Distributions", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 5.4 Propensity Score Matching

### 5.4.1 Run PSM

The `CausalAnalysisAgent` encapsulates the full PSM pipeline: it loads
evaluation results, encodes covariates, fits propensity models, performs 1:1
nearest-neighbour matching via `cKDTree`, computes the ATE with bootstrap
confidence intervals, and runs Rosenbaum sensitivity analysis. Calling
`agent.run()` executes this pipeline for all five treatments and saves
`causal_results.json` to the results directory.

In [ ]:
agent = CausalAnalysisAgent(
    results_dir=str(results_dir),
    plots_dir=str(results_dir / "causal_plots"),
)
causal_results = agent.run()
print(f"\nCompleted analysis for {len(causal_results)} treatments.")

### 5.4.2 Post-Matching Balance

After matching, we re-examine covariate balance to confirm that PSM has reduced
the standardized mean differences below the conventional 0.1 threshold. A
successful match means the treated and matched-control groups are now comparable
on observed covariates, strengthening the causal interpretation of the ATE.
If any covariate remains imbalanced, the corresponding treatment effect should
be interpreted with caution.

In [ ]:
# Re-run matching to extract post-match covariate balance
rng = np.random.default_rng(cfg["random_seed"])
post_smd_rows = []

for tdef in cfg["treatments"]:
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = np.array([i for i, r in enumerate(records) if t_filter(r)])
    c_idx = np.array([i for i, r in enumerate(records) if c_filter(r)])

    if len(t_idx) < cfg["min_treated"] or len(c_idx) < cfg["min_treated"]:
        continue

    all_idx = np.concatenate([t_idx, c_idx])
    T = np.array([1] * len(t_idx) + [0] * len(c_idx))
    X = np.column_stack([cov_arrays[cov][all_idx] for cov in tdef["covariates"]])

    lr = LogisticRegression(C=1.0, max_iter=500, random_state=cfg["random_seed"])
    lr.fit(X, T)
    pscore = lr.predict_proba(X)[:, 1]

    # 1:1 nearest-neighbour matching
    pscore_control = pscore[T == 0]
    pscore_treated = pscore[T == 1]
    tree = cKDTree(pscore_control.reshape(-1, 1))
    _, nn_indices = tree.query(pscore_treated.reshape(-1, 1), k=1)

    # Indices into the all_idx array
    control_positions = np.where(T == 0)[0]
    matched_control_positions = control_positions[nn_indices]
    treated_positions = np.where(T == 1)[0]

    for cov_name in tdef["covariates"]:
        arr = cov_arrays[cov_name]
        cov_treated = arr[all_idx[treated_positions]]
        cov_matched = arr[all_idx[matched_control_positions]]
        smd_val = compute_smd(cov_treated, cov_matched)
        post_smd_rows.append({
            "treatment": tdef["name"],
            "covariate": cov_name,
            "SMD_post_match": round(smd_val, 4),
            "balanced": "Yes" if smd_val < 0.1 else "No",
        })

post_smd_df = pd.DataFrame(post_smd_rows)
print("Post-matching covariate balance (SMD):")
display(post_smd_df)

# Compare pre vs post
if len(post_smd_rows) > 0:
    comparison = smd_df.merge(post_smd_df, on=["treatment", "covariate"], how="inner")
    print("\nPre vs Post matching SMD comparison:")
    display(comparison[["treatment", "covariate", "SMD_pre_match", "SMD_post_match", "balanced"]])

## 5.5 Results

### 5.5.1 ATE Table

The main output of the PSM pipeline is the average treatment effect for each
treatment, accompanied by 95% bootstrap confidence intervals and Rosenbaum
sensitivity bounds. The table below shows whether each effect is statistically
distinguishable from zero and how robust it is to potential unmeasured confounding.
A higher Gamma value indicates the finding would survive larger amounts of
hidden bias.

In [ ]:
# Load saved causal results
causal_path = results_dir / "causal_results.json"
with open(causal_path) as f:
    causal_results_loaded = json.load(f)

ate_rows = []
for r in causal_results_loaded:
    ate_rows.append({
        "Treatment": r["treatment"],
        "Outcome": r["outcome"],
        "ATE": f"{r['ate']:.4f}",
        "95% CI": f"[{r['ci_lower']:.4f}, {r['ci_upper']:.4f}]",
        "Gamma": f"{r['rosenbaum_gamma']:.2f}",
        "n_treated": r["n_treated"],
        "overlap_ok": r["overlap_ok"],
    })

ate_df = pd.DataFrame(ate_rows)
print("Average Treatment Effects (PSM):")
display(ate_df)

### 5.5.2 Interpretation

The following paragraphs translate the numerical ATE estimates into plain-English
findings. Each treatment's effect is contextualised by its confidence interval
width, the Rosenbaum Gamma, and whether propensity overlap was adequate. These
interpretations are intentionally conservative: we flag uncertainty rather than
over-claiming.

In [ ]:
# Auto-generate plain-English interpretation for each treatment
interpretation_templates = {
    "rain": (
        "(95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]). With a Rosenbaum Gamma of {gamma:.2f}, "
        "this finding {robustness}. Overlap status: {overlap}."
    ),
    "night": (
        "Nighttime driving {direction} route completion by {abs_ate:.1f} percentage points "
        "(95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]). The Gamma of {gamma:.2f} suggests the result "
        "{robustness}. Overlap status: {overlap}."
    ),
    "urban": (
        "Urban terrain (Town01/Town03) {direction} collision rate by {abs_ate:.4f} "
        "(95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]). Gamma = {gamma:.2f} indicates the effect "
        "{robustness}. Overlap status: {overlap}."
    ),
    "ppo": (
        "PPO fine-tuning {direction} route completion by {abs_ate:.1f} percentage points "
        "compared to behavior cloning (95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]). "
        "Gamma = {gamma:.2f} -- {robustness}. Overlap status: {overlap}."
    ),
    "driving_style": (
        "Comfort-optimized (chill) PPO {direction} route completion by {abs_ate:.1f} "
        "percentage points versus BC (95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]). "
        "Gamma = {gamma:.2f} -- {robustness}. Overlap status: {overlap}."
    ),
}

for r in causal_results_loaded:
    name = r["treatment"]
    ate = r["ate"]
    direction = "increases" if ate > 0 else "decreases"
    gamma = r["rosenbaum_gamma"]
    robustness = (
        "is robust to moderate hidden bias" if gamma >= 1.5
        else "is sensitive to even small unmeasured confounders" if gamma <= 1.1
        else "has moderate robustness to hidden bias"
    )
    overlap = "adequate" if r["overlap_ok"] else "POOR -- interpret with caution"

    template = interpretation_templates.get(name, "{direction} outcome by {abs_ate:.4f}")
    text = template.format(
        direction=direction,
        abs_ate=abs(ate) * 100,
        ci_lo=r["ci_lower"],
        ci_hi=r["ci_upper"],
        gamma=gamma,
        robustness=robustness,
        overlap=overlap,
    )
    print(f"\n{name.upper()}:")
    print(f"  {text}")

## 5.6 Sensitivity Analysis

### 5.6.1 Rosenbaum Bounds Plot

The Rosenbaum sensitivity parameter Gamma represents the smallest magnitude of
hidden bias (unmeasured confounding) that would make a treatment effect
statistically insignificant at p = 0.05. A Gamma of 1.0 means the finding is
already non-significant; higher values indicate greater robustness. For
instance, Gamma = 2.0 means the result would remain significant even if an
unmeasured confounder doubled the odds of treatment assignment.

In [ ]:
treatment_names = [r["treatment"] for r in causal_results_loaded]
gamma_values = [r["rosenbaum_gamma"] for r in causal_results_loaded]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    treatment_names, gamma_values,
    color="steelblue", edgecolor="black", alpha=0.8,
)

# Annotate each bar with its Gamma value
for bar, val in zip(bars, gamma_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
        f"{val:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

# p = 0.05 threshold annotation: Gamma = 1.0 is the minimum (no hidden bias)
ax.axhline(
    y=1.0, color="red", linestyle="--", linewidth=1.5,
    label="Gamma = 1.0 (no hidden bias)",
)
ax.text(
    len(treatment_names) - 0.5, 1.05,
    "p = 0.05 threshold", color="red", fontsize=9, ha="right",
)

ax.set_ylabel("Rosenbaum Gamma")
ax.set_xlabel("Treatment")
ax.set_title("Sensitivity to Unmeasured Confounding (Rosenbaum Bounds)")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

### 5.6.2 Overlap Diagnostics

Adequate overlap between treated and control propensity score distributions is a
prerequisite for valid PSM. When overlap is poor, matched pairs may be drawn
from distant regions of the propensity space, violating common support. This
section reports the overlap status and propensity score ranges for each
treatment to flag any problematic cases.

In [ ]:
for tdef in cfg["treatments"]:
    t_filter = build_filter(tdef["treated_condition"])
    c_filter = build_filter(tdef["control_condition"])
    t_idx = np.array([i for i, r in enumerate(records) if t_filter(r)])
    c_idx = np.array([i for i, r in enumerate(records) if c_filter(r)])

    name = tdef["name"]

    if len(t_idx) < cfg["min_treated"] or len(c_idx) < cfg["min_treated"]:
        print(f"\n{name}: SKIPPED (insufficient data: {len(t_idx)} treated, {len(c_idx)} control)")
        continue

    all_idx = np.concatenate([t_idx, c_idx])
    T = np.array([1] * len(t_idx) + [0] * len(c_idx))
    X = np.column_stack([cov_arrays[cov][all_idx] for cov in tdef["covariates"]])

    lr = LogisticRegression(C=1.0, max_iter=500, random_state=cfg["random_seed"])
    lr.fit(X, T)
    pscore = lr.predict_proba(X)[:, 1]

    ps_t = pscore[T == 1]
    ps_c = pscore[T == 0]

    # Overlap check (mirrors agent._check_overlap)
    overlap_range = min(ps_t.max(), ps_c.max()) - max(ps_t.min(), ps_c.min())
    frac_boundary = (np.mean(ps_t > 0.9) + np.mean(ps_c < 0.1)) / 2
    ok = overlap_range >= 0 and frac_boundary < 0.5

    # Find matching result from agent output
    result_match = next((r for r in causal_results_loaded if r["treatment"] == name), None)
    overlap_status = result_match["overlap_ok"] if result_match else ok

    print(f"\n{name}:")
    print(f"  Overlap OK     : {overlap_status}")
    print(f"  Treated  range : [{ps_t.min():.4f}, {ps_t.max():.4f}]")
    print(f"  Control  range : [{ps_c.min():.4f}, {ps_c.max():.4f}]")
    print(f"  Overlap width  : {overlap_range:.4f}")
    print(f"  Frac near boundary: {frac_boundary:.4f}")

## 5.7 Limitations and Assumptions

### 5.7.1 Unobserved Confounders

Propensity score matching controls for observed covariates but cannot account
for confounders that are not measured or encoded. In the CARLA simulator,
potential unobserved confounders include spawn-point geometry (e.g., starting on
a hill versus a straight road), NPC vehicle behavior randomness (which varies
between runs even under the same traffic density), and network or GPU latency
spikes that could cause frame drops during inference. Additionally, route
topology within a given town is not explicitly controlled: two episodes in the
same town may encounter very different intersection counts or curve radii. The
Rosenbaum sensitivity bounds reported above provide some reassurance by
quantifying how large a hidden bias would need to be to overturn each finding.

### 5.7.2 PSM Validity Conditions

PSM estimates are most reliable when several conditions are satisfied: the
propensity score distributions of treated and control groups overlap
substantially, post-matching standardized mean differences on all covariates
fall below 0.1, and no propensity scores cluster near 0 or 1 (which would
indicate near-deterministic treatment assignment). When these conditions fail,
the matched comparison may be driven by a small number of extreme observations.
In such cases, alternative estimators (e.g., inverse probability weighting with
trimming or doubly-robust estimators) may be preferable. The overlap diagnostics
in Section 5.6.2 should be checked before drawing policy conclusions from any
treatment with `overlap_ok = False`.

### 5.7.3 Simulator vs Real-World Generalization

All causal conclusions in this notebook are derived from the CARLA simulator,
which, despite its fidelity, differs from real driving in several important
ways. CARLA's physics engine simplifies tire-road interactions, aerodynamics,
and sensor noise relative to physical vehicles. The traffic variety is limited
to a fixed set of NPC vehicle blueprints with scripted or autopilot-driven
behavior, and the evaluation grid does not include pedestrians, cyclists, or
construction zones. Weather effects in CARLA approximate but do not perfectly
reproduce real-world visibility degradation or road surface changes. Therefore,
while the directional findings (e.g., rain hurts performance, PPO improves over
BC) are likely to transfer, the exact magnitudes of the estimated treatment
effects should not be taken as predictions for real-world deployment.